In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import re

In [ ]:
# ---- CONFIG ----
# csv_path = "synthetic_results_by_classes.csv"   # adjust path if needed
csv_path = "results_cleaned.csv"   # adjust path if needed
out_path = "nll_brier_vs_classes_highlight_conformal.pdf"
# ----------------

df = pd.read_csv(csv_path)

In [ ]:
# df.loc[df["Calibrator"] == "isotonic", "Calibrator"] = "Isotonic"
# df.loc[df["Calibrator"] == "temp_scaling", "Calibrator"] = "Temp. Scaling"
# df.loc[df["Calibrator"] == "uncalibrated", "Calibrator"] = "Base"
# df.loc[df["Calibrator"] == "cnfrml_mass_thrsh:a=0.1,sс.tp=one,sс.trnf=iden", "Calibrator"] = "Ours, MR, I, 0.1"


In [ ]:
df

In [ ]:
# df.to_csv("results_cleaned.csv", index=False)

In [ ]:
import matplotlib

def pretty_matplotlib_config(
    fontsize=15,
    legend_fontsize=None,
    legend_title_fontsize=None,
    axes_titlesize=None,
    axes_labelsize=None,
    tick_labelsize=None,
    suptitle_size=None,
):
    rc = matplotlib.rcParams
    rc['pdf.fonttype'] = 42
    rc['ps.fonttype'] = 42
    rc['text.usetex'] = True

    # Base font
    rc['font.size'] = fontsize

    # Legend
    rc['legend.fontsize'] = legend_fontsize if legend_fontsize is not None else 0.8 * fontsize
    rc['legend.title_fontsize'] = (
        legend_title_fontsize if legend_title_fontsize is not None else rc['legend.fontsize']
    )

    # Axes titles & labels
    rc['axes.titlesize'] = axes_titlesize if axes_titlesize is not None else 1.1 * fontsize
    rc['axes.labelsize'] = axes_labelsize if axes_labelsize is not None else 0.95 * fontsize

    # Tick labels
    rc['xtick.labelsize'] = tick_labelsize if tick_labelsize is not None else 0.85 * fontsize
    rc['ytick.labelsize'] = tick_labelsize if tick_labelsize is not None else 0.85 * fontsize

    # (optional) Figure suptitle
    if suptitle_size is not None:
        rc['figure.titlesize'] = suptitle_size  # affects plt.suptitle

In [ ]:
# example
pretty_matplotlib_config(
    fontsize=35,
    legend_fontsize=25,
    axes_titlesize=40,   # title of each axes
    axes_labelsize=40,   # x/y labels
    tick_labelsize=40,   # tick labels
)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.lines import Line2D
import unicodedata, re

# ---------- Config ----------
csv_path = "synthetic_results_by_classes.csv"      # set your path if different
out_png = "stacked_nll_brier__cmce_clean.png"
out_pdf = "stacked_nll_brier__cmce_clean.pdf"

# ---------- Load ----------
df = pd.read_csv(csv_path)

def extract_classes(name: str) -> int:
    m = re.search(r"synthetic_(\d+)_grid", str(name))
    return int(m.group(1)) if m else None

if "n_classes" not in df.columns:
    df["n_classes"] = df["Dataset"].apply(extract_classes)
else:
    df["n_classes"] = df["n_classes"].apply(lambda x: int(x) if pd.notnull(x) else x)

df = df.sort_values(["n_classes", "Calibrator"]).reset_index(drop=True)

# ---------- Label helpers ----------
def clean_ascii(s: str) -> str:
    # replace Cyrillic lookalikes and strip non-ASCII
    repl = {
        "\u0441": "c", "\u0430": "a", "\u043e": "o", "\u0435": "e",
        "\u0440": "p", "\u0445": "x", "\u0412": "B", "\u0421": "C",
        "\u041E": "O", "\u0410": "A", "\u0420": "P", "\u0425": "X", "\u0415": "E",
    }
    for k, v in repl.items():
        s = s.replace(k, v)
    s = unicodedata.normalize("NFKD", s)
    return s.encode("ascii", "ignore").decode("ascii")

def short_label(raw: str) -> str:
    r = str(raw)
    cr = clean_ascii(r)
    if "cnfrml_mass_thrsh" in cr or "Ours, MR, I, 0.1" in r or "conformal" in cr:
        return "Ours, MR, I, 0.1"
    if "temp_scaling" in cr:
        return "Temp. Scaling"
    if "isotonic" in cr:
        return "Isotonic"
    if "uncalibrated" in cr:
        return "Uncalibrated"
    return cr[:40] + ("..." if len(cr) > 40 else "")

def is_ours(s: str) -> bool:
    s = str(s)
    return ("cnfrml_mass_thrsh" in s) or ("Ours, MR, I, 0.1" in s) or ("conformal" in s)

# ---------- Figure ----------
fig = plt.figure(figsize=(12, 10), constrained_layout=True)
gs = GridSpec(2, 1, height_ratios=[3, 1.15], hspace=0.06, figure=fig)

# Top axes: NLL (left) + Brier (right)
ax_nll = fig.add_subplot(gs[0, 0])
ax_brier = ax_nll.twinx()
# Bottom axis: CMCE, shares x with top
ax_cmce = fig.add_subplot(gs[1, 0], sharex=ax_nll)

# Log scales
for ax in (ax_nll, ax_brier, ax_cmce):
    ax.set_xscale("log")
    ax.set_yscale("log")

# Labels
ax_nll.set_ylabel("NLL")
ax_brier.set_ylabel("Brier score")
ax_cmce.set_ylabel("CMCE")
ax_cmce.set_xlabel("Number of classes")

# Hide top x tick labels (shared x)
plt.setp(ax_nll.get_xticklabels(), visible=False)

# Plot lines (emphasize "Ours")
calib_handles, calib_names = [], []
for calib, sub in df.groupby("Calibrator"):
    sub = sub.sort_values("n_classes")
    label = short_label(calib)
    highlight = is_ours(calib)
    lw = 2.6 if highlight else 1.1
    ms_top = 6.5 if highlight else 3.2
    ms_bot = 6.0 if highlight else 3.0
    alpha = 1.0 if highlight else 0.7
    z = 6 if highlight else 3

    # NLL (left/top)
    h1, = ax_nll.plot(
        sub["n_classes"], sub["nll"],
        linestyle="-", marker="o",
        linewidth=lw, markersize=ms_top, alpha=alpha, zorder=z,
        label=label
    )
    # Brier (right/top)
    ax_brier.plot(
        sub["n_classes"], sub["brier_score"],
        linestyle="--", marker="^",
        linewidth=lw, markersize=ms_top, alpha=alpha, zorder=z
    )
    # CMCE (bottom)
    ax_cmce.plot(
        sub["n_classes"], sub["cmce"],
        linestyle=":", marker="s",
        linewidth=lw, markersize=ms_bot, alpha=alpha, zorder=z
    )

    calib_handles.append(h1)     # one handle per calibrator in legend
    calib_names.append(label)

# Grids
ax_nll.grid(True, which="both", linewidth=0.6, alpha=0.12)
ax_cmce.grid(True, which="both", linewidth=0.6, alpha=0.12)

# Global legend above (no overlap)
# fig.legend(
#     calib_handles, calib_names,
#     loc="upper center"
# )
fig.legend(
    calib_handles, calib_names,
    loc="upper center",            # anchor point on the legend box
    bbox_to_anchor=(0.3, 0.95),   # (x, y) in figure coords; try 0.35–0.45
)

# Slight x padding
xmin = max(1, df["n_classes"].min())
xmax = df["n_classes"].max()
ax_nll.set_xlim(xmin * 0.9, xmax * 1.1)

# Save
fig.savefig(out_png, dpi=300, bbox_inches="tight")
fig.savefig(out_pdf, dpi=300, format="pdf", bbox_inches="tight")
print(f"Saved PNG to: {out_png}")
print(f"Saved PDF to: {out_pdf}")


In [ ]:
import sys
sys.path.insert(0, "../")

from caliblab.datasets import Synthetic2DClassifier
from torch.utils.data import Dataset
import torch
import numpy as np

In [ ]:
class Synth2DDataset(Dataset):
    """
    Pre-samples a fixed split so you can use standard DataLoaders.
    """
    def __init__(self, ds: Synthetic2DClassifier, n_samples: int, seed: int = 0):
        rng = np.random.default_rng(seed)
        # clone a dataset instance with a controlled RNG for reproducibility
        self.ds = ds
        self.x, self.y, _ = ds.sample(n_samples)
        self.x = self.x.astype(np.float32)
        self.y = self.y.astype(np.int64)

    def __len__(self):
        return self.x.shape[0]

    def __getitem__(self, idx):
        return torch.from_numpy(self.x[idx]), torch.tensor(self.y[idx])

In [ ]:
ds_ = Synthetic2DClassifier(n_classes=49)
ds = Synth2DDataset(ds_, n_samples=10000)

In [ ]:
x = ds.x
y = ds.y

In [ ]:
plt.figure(figsize=(12, 10), dpi=150)
plt.scatter(x[:, 0], x[:, 1], c=y, cmap="tab20", s=18, alpha=0.85, edgecolor="k", linewidth=0.25)
plt.gca().set_aspect("equal", adjustable="datalim")
plt.grid(True, which="both", linewidth=0.6, alpha=0.15)
plt.tight_layout()
plt.savefig("synthetic_data_example.png", format="png", bbox_inches="tight")